In [2]:
# Exercise 8.(Binary Addition)
# a) Define a Python function h_add(a0,b0) which adds two bits a0, b0 and returns
#  the sum bit s as well as the carry bit c. The corresponding logical terms are
#  s =a0⊕b0, c = a0&b0.
# b) Define a Python function f_add(a1,b1,c1) which adds two bits a1, b1 and the
#   carry bit c1 and returns the sum bit s as well as the carry bit c. The logical
#   terms are s = (a1 ⊕b1)⊕c1, c = (a1&b1)∨((a1 ⊕b1)&c1).
# c) Use the logical terms from a) and b) to obtain logical terms for adding binary
#    numbers with two, three and four bits.
# d) Define a Python function bin_add([an,...,a0],[bn,...,b0]) which adds two 
#    binary numbers with bits [an,...,a0], [bn,...,b0] and returns a list of sum bits
#    [sn, . . . , s0] and the carry bit c.
# e) Define Python functions DecToBin() and BinToDec() to covert a nonnegative
#    integer to a list of bits (the lower bit is to the right), respectively vice versa. Use
#    the functions together with bin_add() to add nonnegative integers

In [5]:
# a) and b)
from sympy.logic.boolalg import Xor
from sympy import symbols

def h_add(a0, b0):
    a0, b0 = bool(a0), bool(b0)
    s = Xor(a0, b0)
    c = a0 & b0
    return bool(s), bool(c)

def f_add(a1, b1, c1):
    a1, b1, c1 = bool(a1), bool(b1), bool(c1)
    s = Xor(Xor(a1, b1), c1)
    c = (a1 & b1) | (Xor(a1, b1) & c1)
    return bool(s), bool(c)

# Test h_add
print("Half adder:")
print(f"  h_add(0,0) = {h_add(False, False)}")  # s=0, c=0
print(f"  h_add(0,1) = {h_add(False, True)}")   # s=1, c=0
print(f"  h_add(1,0) = {h_add(True,  False)}")  # s=1, c=0
print(f"  h_add(1,1) = {h_add(True,  True)}")   # s=0, c=1

# Test f_add
print("\nFull adder:")
print(f"  f_add(0,0,0) = {f_add(False, False, False)}")  # s=0, c=0
print(f"  f_add(1,1,0) = {f_add(True,  True,  False)}")  # s=0, c=1
print(f"  f_add(1,1,1) = {f_add(True,  True,  True)}")   # s=1, c=1

Half adder:
  h_add(0,0) = (False, False)
  h_add(0,1) = (True, False)
  h_add(1,0) = (True, False)
  h_add(1,1) = (False, True)

Full adder:
  f_add(0,0,0) = (False, False)
  f_add(1,1,0) = (False, True)
  f_add(1,1,1) = (True, True)


In [6]:
# c)
# ── 2-bit addition: a = [a1,a0], b = [b1,b0] ────────────────────────
def add2(a, b):
    s0, c0 = h_add(a[1], b[1])           # LSB: half adder
    s1, c1 = f_add(a[0], b[0], c0)       # next bit: full adder
    return [s1, s0], c1

# ── 3-bit addition ───────────────────────────────────────────────────
def add3(a, b):
    s0, c0 = h_add(a[2], b[2])
    s1, c1 = f_add(a[1], b[1], c0)
    s2, c2 = f_add(a[0], b[0], c1)
    return [s2, s1, s0], c2

# ── 4-bit addition ───────────────────────────────────────────────────
def add4(a, b):
    s0, c0 = h_add(a[3], b[3])
    s1, c1 = f_add(a[2], b[2], c0)
    s2, c2 = f_add(a[1], b[1], c1)
    s3, c3 = f_add(a[0], b[0], c2)
    return [s3, s2, s1, s0], c3

# Test: 2-bit  01 + 01 = 10  (1 + 1 = 2)
print(f"add2([0,1],[0,1]) = {add2([0,1],[0,1])}")   # [1,0], carry 0
# Test: 3-bit  011 + 011 = 110  (3 + 3 = 6)
print(f"add3([0,1,1],[0,1,1]) = {add3([0,1,1],[0,1,1])}")

add2([0,1],[0,1]) = ([True, False], False)
add3([0,1,1],[0,1,1]) = ([True, True, False], False)


In [7]:
# d)
def bin_add(a, b):
    """
    Add two binary numbers represented as lists of bits.
    MSB is leftmost (index 0), LSB is rightmost (index -1).
    Returns (sum_bits, carry).
    """
    assert len(a) == len(b), "Bit lists must have equal length"
    
    n = len(a)
    s_bits = [None] * n
    
    # Start with half adder on LSB (rightmost)
    s_bits[n-1], carry = h_add(a[n-1], b[n-1])
    
    # Full adders for remaining bits, right to left
    for i in range(n-2, -1, -1):
        s_bits[i], carry = f_add(a[i], b[i], carry)
    
    return s_bits, carry

# Test
print(f"bin_add([1,0,1],[0,1,1]) = {bin_add([1,0,1],[0,1,1])}")  # 5+3=8 → [0,0,0], carry 1

bin_add([1,0,1],[0,1,1]) = ([False, False, False], True)


In [8]:
# e)
def DecToBin(n, bits=None):
    """Convert nonnegative integer to list of bits (MSB left, LSB right)."""
    if n == 0:
        return [0] if bits is None else [0] * bits
    
    result = []
    while n > 0:
        result.append(n % 2)
        n //= 2
    result.reverse()  # MSB first
    
    # Pad to desired length if specified
    if bits is not None:
        result = [0] * (bits - len(result)) + result
    
    return result

def BinToDec(bits):
    """Convert list of bits (MSB left, LSB right) to integer."""
    result = 0
    for b in bits:
        result = result * 2 + int(b)
    return result

def add_integers(x, y):
    """Add two nonnegative integers using bin_add."""
    # Determine bit length needed
    bits = max(len(DecToBin(x)), len(DecToBin(y)))
    
    a = DecToBin(x, bits)
    b = DecToBin(y, bits)
    
    s_bits, carry = bin_add(a, b)
    
    # Prepend carry as MSB if needed
    if carry:
        s_bits = [True] + s_bits
    
    result = BinToDec(s_bits)
    print(f"  {x} + {y} = {result}  "
          f"(bin: {a} + {b} = {s_bits}, carry={int(carry)})")
    return result

# ── Tests ────────────────────────────────────────────────────────────
print("DecToBin / BinToDec:")
print(f"  DecToBin(13)    = {DecToBin(13)}")         # [1,1,0,1]
print(f"  BinToDec([1,1,0,1]) = {BinToDec([1,1,0,1])}")  # 13

print("\nInteger addition via bin_add:")
add_integers(5,  3)    # 8
add_integers(7,  7)    # 14
add_integers(13, 6)    # 19
add_integers(15, 1)    # 16  — carry propagates all the way

DecToBin / BinToDec:
  DecToBin(13)    = [1, 1, 0, 1]
  BinToDec([1,1,0,1]) = 13

Integer addition via bin_add:
  5 + 3 = 8  (bin: [1, 0, 1] + [0, 1, 1] = [True, False, False, False], carry=1)
  7 + 7 = 14  (bin: [1, 1, 1] + [1, 1, 1] = [True, True, True, False], carry=1)
  13 + 6 = 19  (bin: [1, 1, 0, 1] + [0, 1, 1, 0] = [True, False, False, True, True], carry=1)
  15 + 1 = 16  (bin: [1, 1, 1, 1] + [0, 0, 0, 1] = [True, False, False, False, False], carry=1)


16